# 118 — Memoria, contexto y continuidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

El LLM es una función sin estado: solo "recuerda" su ventana de contexto. La
continuidad se construye con tres almacenes:

- **Contexto (memoria de trabajo):** instrucciones + plan + ternas recientes. Volátil,
  cara (se paga por token en CADA llamada), limitada.
- **Memoria persistente:** *episódica* (qué pasó: trazas) y *semántica* (hechos
  destilados con procedencia). Entra al contexto por recuperación selectiva.
- **Checkpoint:** instantánea del estado del bucle (plan + variables + log de efectos
  aplicados) en un punto consistente; su contrato es reanudar SIN repetir efectos.

### 📉 Gestionar el contexto que crece

Estrategias: **compactar** (ternas viejas → resumen estructurado; con pérdida — errores
y decisiones se conservan textuales), **externalizar** (detalle → archivo/BD + puntero
en contexto) y **seleccionar** (recuperar solo lo relevante al paso actual, parte 08).

```text
contexto = instrucciones (fijo) + plan (siempre visible)
         + resumen de lo hecho + últimas K ternas + recuperado bajo demanda
```

Riesgo propio de la memoria: lo persistido vuelve a entrar en sesiones futuras — un
dato envenenado hoy es una "verdad recordada" mañana. Etiquetar procedencia y curar
(caducidad, corrección) no es opcional.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("agent", seed=118)
show(result)


## Reflexión

1. El laboratorio termina en dos pasos y no necesita checkpoint. ¿En qué momento exacto
   de una tarea de 40 pasos tomarías checkpoints y por qué "nunca en mitad de un
   efecto"?
2. La compactación es una operación con pérdida. ¿Qué dos tipos de contenido deben
   conservarse textuales aunque todo lo demás se resuma, y qué error induce omitirlos?
3. ¿Por qué la memoria semántica recuperada debe tratarse como dato de baja procedencia
   aunque la haya escrito el propio agente?